In [1]:
import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse

RAW_PATH = r"E:\file_main\Major_2\data\raw"
OUT_PATH = r"E:\file_main\Major_2\data\processed"

PHISHTANK_FILE = f"{RAW_PATH}\\phishtank.csv"
TRANCO_FILE = f"{RAW_PATH}\\tranco_3Q88L.csv"

PHISH_LABEL = 1
LEGIT_LABEL = 0

SAMPLE_SIZE = 45000
RANDOM_STATE = 42


In [16]:
phish_df = pd.read_csv(PHISHTANK_FILE)
tranco_df = pd.read_csv(TRANCO_FILE, header=None, names=["rank", "domain"])

#sanity check
print(phish_df.shape)
print(tranco_df.shape)

#Keeping only verified phishing links
phish_df = phish_df[
    (phish_df["verified"] == "yes") &
    (phish_df["online"] == "yes")
]
print("Number of enteries after removing unverified:",phish_df.shape)

(46660, 8)
(4242838, 2)
Number of enteries after removing unverified: (46660, 8)


In [18]:
#Cleaning Phishing URLs
def clean_url(url):
    if not isinstance(url, str):
        return None
    url = url.strip().lower()
    if not url.startswith(("http://", "https://")):
        return None
    return url

phish_urls = (
    phish_df["url"]
    .apply(clean_url)
    .dropna()
    .drop_duplicates()
)

phish_clean = pd.DataFrame({
    "url": phish_urls,
    "label": PHISH_LABEL
})

print(phish_df.shape)

(46660, 8)


In [20]:
#Standardizing Tranco URLs
def domain_to_url(domain):
    domain = domain.strip().lower()
    return f"https://www.{domain}"

tranco_urls = tranco_df["domain"].apply(domain_to_url)
print(tranco_df.shape)

(4242838, 2)


In [21]:
tranco_sampled = tranco_urls.sample(
    n=SAMPLE_SIZE,
    random_state=RANDOM_STATE
)

legit_clean = pd.DataFrame({
    "url": tranco_sampled.values,
    "label": LEGIT_LABEL
})


In [22]:
tranco_sampled = tranco_urls.sample(
    n=SAMPLE_SIZE,
    random_state=RANDOM_STATE
)

legit_clean = pd.DataFrame({
    "url": tranco_sampled.values,
    "label": LEGIT_LABEL
})

In [24]:
final_df = pd.concat([phish_clean, legit_clean], ignore_index=True)
final_df = final_df.drop_duplicates(subset="url")
final_df = final_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

In [25]:
print(final_df["label"].value_counts())
print(final_df.shape)

assert final_df.isnull().sum().sum() == 0
assert final_df["url"].str.startswith(("http://", "https://")).all()


label
1    46652
0    45000
Name: count, dtype: int64
(91652, 2)


In [26]:
import os
os.makedirs(OUT_PATH, exist_ok=True)

OUTPUT_FILE = f"{OUT_PATH}\\phishing_urls_final.csv"
final_df.to_csv(OUTPUT_FILE, index=False)

print(final_df.shape)

(91652, 2)
